# LSLC: Layered Spatial Latent Caching — Google Colab MVE

Run **Milestone 1–4** on **Google Colab T4 (15 GB)**, **L4 (24 GB)**, or **A100 (40 GB)** using frozen **`Qwen/Qwen3-1.7B`** (default) or **`Qwen/Qwen3-4B`** (`--load-in-4bit` on T4 / native BF16 on L4+).

**Before running long training:** always run unit tests and the mandatory **20-step smoke test** first.

In [ ]:
import torch

print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.2f} GiB | BF16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No CUDA GPU detected. Only dry-run mock checks will run fast.")

In [ ]:
# Clone or mount your lslc-research repo here, then install dependencies
# !git clone <YOUR_REPO_URL> /content/lslc-research
%cd /content/lslc-research
!pip install -q -r requirements-colab.txt

In [ ]:
# Step 1 (0 CU): Run Milestone 1 Unit Tests & Analytical Memory Table
!python3 -m unittest -v tests/test_lslc_shapes.py
!python3 scripts/profile_vram.py --analytical-only

In [ ]:
# Step 2: Mandatory 20-Step Smoke Test on Qwen/Qwen3-1.7B
# (For Qwen/Qwen3-4B on 15 GB T4, add: --model Qwen/Qwen3-4B --load-in-4bit)
!python3 scripts/train_lslc.py \
  --model Qwen/Qwen3-1.7B \
  --track 4 \
  --smoke-test \
  --num-pages 4 \
  --output-dir checkpoints/smoke_track4

In [ ]:
# Step 3 (Milestone 2): Track 1 (Dense Only) vs Track 2 (+ Sparse Exact Bank)
!python3 scripts/train_lslc.py --model Qwen/Qwen3-1.7B --track 1 --max-steps 500 --num-pages 4 --output-dir checkpoints/m2_track1
!python3 scripts/eval_lslc.py --model Qwen/Qwen3-1.7B --track 1 --checkpoint checkpoints/m2_track1/lslc_weights.pt --output-dir evals/track1_dense

!python3 scripts/train_lslc.py --model Qwen/Qwen3-1.7B --track 2 --max-steps 500 --num-pages 4 --output-dir checkpoints/m2_track2
!python3 scripts/eval_lslc.py --model Qwen/Qwen3-1.7B --track 2 --checkpoint checkpoints/m2_track2/lslc_weights.pt --output-dir evals/track2_exact

In [ ]:
# Step 4 (Milestone 3): Multi-Page Book (Track 3) & Living-Book Delta-Updates (Track 4) + Live CUDA Profiler
!python3 scripts/train_lslc.py --model Qwen/Qwen3-1.7B --track 4 --max-steps 600 --num-pages 8 --output-dir checkpoints/m3_track4
!python3 scripts/eval_lslc.py --model Qwen/Qwen3-1.7B --track 4 --num-pages 8 --checkpoint checkpoints/m3_track4/lslc_weights.pt --output-dir evals/track4_delta
!python3 scripts/profile_vram.py --model Qwen/Qwen3-1.7B --context-lengths 4096 32768 --output-json evals/vram_ttft_profile.json